# Akshara Timestamps
Standalone notebook for CTC / TDT inference with word-level timestamps.  
Blank penalty and timestamp logic mirrors `model_repo/nemo_hybrid/1/model.py`.

In [ ]:
import numpy as np
import pandas as pd
import torch
import soundfile as sf
from typing import List, Dict, Optional, Tuple
from nemo.collections.asr.models import EncDecHybridRNNTCTCBPEModel
import re
import torchaudio
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from jiwer import cer, wer
import torch.nn.functional as F


[NeMo W 2026-05-26 10:16:09 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
[NeMo W 2026-05-26 10:16:10 nemo_logging:364] /home/shourya_1_nobroker_in/miniconda3/envs/myenv/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)
    


In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
INPUT_CSV              = "/home/ubuntu/Pavan/acoustic_fusion/final.csv"
OUTPUT_CSV             = "grid_search_results.csv"
MAX_ASSISTANT_DURATION = 17
ASSISTANT_TRIM_MODE    = "end"
MAX_COMBINED_DURATION  = 15.0    # seconds — hard cap on assistant+silence+user
NUM_WORKERS            = 12
STRATEGY               = "tdt"
ALPHA                  = 0.5

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
NEMO_MODEL_PATH = "/home/shourya_1_nobroker_in/shourya/akshara_tdt.nemo"   # adjust as needed
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Load model ────────────────────────────────────────────────────────────────
model = EncDecHybridRNNTCTCBPEModel.restore_from(NEMO_MODEL_PATH, map_location=DEVICE)
model.eval()
model.requires_grad_(False)

model.cfg.decoding.greedy.preserve_alignments = True                                                                                                                                                      
model.cfg.decoding.compute_timestamps = True        # ← this is the missing line                                                                                                                        
model.change_decoding_strategy(model.cfg.decoding)

# Disable fuse_loss_wer — training-only optimisation, causes issues at inference
if getattr(model.joint, "_fuse_loss_wer", False):
    model.joint._fuse_loss_wer = False

# ── Derived constants ─────────────────────────────────────────────────────────
BLANK_ID      = getattr(model.ctc_decoding, "blank_id", model.tokenizer.vocab_size)
VOCAB_SIZE    = model.tokenizer.vocab_size
SAMPLE_RATE   = model.cfg.preprocessor.sample_rate
# frame_shift_s: encoder output frame → seconds  (same formula as model.py)
FRAME_SHIFT_S = model.cfg.preprocessor.window_stride * model.cfg.encoder.subsampling_factor
WORD_BOUNDARY = "\u2581"   # SentencePiece word-start marker (▁)
_joint_layer  = model.joint.joint_net[-1]
model.joint.joint_net[2]     # nn.Linear whose bias we patch for TDT blank_penalty

print(f"Model loaded  |  sample_rate={SAMPLE_RATE}  blank_id={BLANK_ID}  "
      f"vocab_size={VOCAB_SIZE}  frame_shift={FRAME_SHIFT_S*1000:.1f} ms")

[NeMo I 2026-04-27 03:25:36 mixins:184] Tokenizer SentencePieceTokenizer initialized with 8192 tokens


[NeMo W 2026-04-27 03:25:45 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /home/ubuntu/Vallabh/conformer/parakeet_tdt_training/cache_v3/audio/training/Parakeet_Hybrid/manifests/train_manifest_9lang_filtered_cleaned.jsonl
    use_lhotse: false
    sample_rate: 16000
    batch_size: 25
    shuffle: true
    num_workers: 12
    pin_memory: true
    use_start_end_token: false
    trim_silence: false
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: 32
    augmentor:
      speed:
        prob: 0.4
        sr: 16000
        min_speed_rate: 0.95
        max_speed_rate: 1.15
        num_rates: -1
        resample_type: kaiser_fast
    
[NeMo W 2026-04-27 03:25:45 modelPT:195] If you in

[NeMo I 2026-04-27 03:25:48 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-04-27 03:25:48 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-04-27 03:25:50 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-04-27 03:25:51 save_restore_connector:285] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /home/ubuntu/Pavan/acoustic_fusion/akshara_tdt.nemo.
[NeMo I 2026-04-27 03:25:51 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-04-27 03:25:51 hybrid_rnnt_ctc_bpe_models:479] Changed decoding s

In [4]:
# ── Timestamp helpers (mirrors model.py) ──────────────────────────────────────

def _compute_ctc_word_timestamps(
    log_probs_np: np.ndarray,   # (T, vocab+1)  — already blank-penalty-adjusted
) -> List[Dict]:
    """
    CTC greedy argmax → collapse repeated tokens + blanks → group into words.
    frame_shift_s converts encoder frame index to seconds.
    Matches model.py _compute_ctc_word_timestamps() exactly.
    """
    raw_ids = np.argmax(log_probs_np, axis=-1).tolist()

    # CTC collapse: (token_id, run_start_frame)
    collapsed: List[Tuple[int, int]] = []
    prev_tid = -1
    run_start = 0
    for t_idx, tid in enumerate(raw_ids):
        if tid != prev_tid:
            if prev_tid != -1 and prev_tid != BLANK_ID and prev_tid < VOCAB_SIZE:
                collapsed.append((prev_tid, run_start))
            run_start = t_idx
            prev_tid  = tid
    if prev_tid != -1 and prev_tid != BLANK_ID and prev_tid < VOCAB_SIZE:
        collapsed.append((prev_tid, run_start))

    if not collapsed:
        return []

    unique_ids = list({tid for tid, _ in collapsed})
    piece_map  = {tid: (model.tokenizer.ids_to_tokens([tid]) or [""])[0] for tid in unique_ids}

    words: List[Dict] = []
    cur_pieces: List[str] = []
    cur_start_emission: Optional[int] = None
    cur_end_emission:   Optional[int] = None
    prev_word_end_emission: Optional[int] = None

    def _flush():
        if not cur_pieces or cur_start_emission is None:
            return
        text = "".join(cur_pieces).lstrip(WORD_BOUNDARY)
        if not text:
            return
        s = prev_word_end_emission if prev_word_end_emission is not None else max(0, cur_start_emission - 1)
        words.append({
            "word":    text,
            "start_s": round(s                * FRAME_SHIFT_S, 3),
            "end_s":   round(cur_end_emission * FRAME_SHIFT_S, 3),
        })

    for tid, frame in collapsed:
        piece = piece_map.get(tid, "")
        if piece.startswith(WORD_BOUNDARY) and cur_pieces:
            _flush()
            prev_word_end_emission = cur_end_emission
            cur_pieces = []
            cur_start_emission = None
            cur_end_emission   = None
        cur_pieces.append(piece)
        if cur_start_emission is None:
            cur_start_emission = frame
        cur_end_emission = frame

    _flush()
    return words


def _hypothesis_to_word_timestamps(hypothesis) -> List[Dict]:
    """
    Extract word timestamps from an RNNT/TDT Hypothesis object.
    hypothesis.timestamp holds per-token encoder-frame indices.
    Matches model.py _hypothesis_to_word_timestamps() exactly.
    """
    y_seq = hypothesis.y_sequence
    if isinstance(y_seq, torch.Tensor):
        y_seq = y_seq.tolist()
    elif hasattr(y_seq, "tolist"):
        y_seq = y_seq.tolist()

    tss = (hypothesis.timestep if hasattr(hypothesis, "timestep")
           else getattr(hypothesis, "timestamp", None))
    if isinstance(tss, dict):
        tss = tss.get("timestep", [])
    if isinstance(tss, torch.Tensor):
        tss = tss.tolist()
    elif hasattr(tss, "tolist"):
        tss = tss.tolist()

    if not y_seq or not tss or len(y_seq) != len(tss):
        return []

    token_strs = model.tokenizer.ids_to_tokens(list(y_seq))

    words: List[Dict] = []
    cur_pieces: List[str] = []
    cur_frames: List[int] = []

    for tstr, frame in zip(token_strs, tss):
        frame = int(frame)
        if tstr.startswith(WORD_BOUNDARY) and cur_pieces:
            text = "".join(cur_pieces).lstrip(WORD_BOUNDARY)
            if text:
                words.append({
                    "word":    text,
                    "start_s": round(min(cur_frames)       * FRAME_SHIFT_S, 3),
                    "end_s":   round((max(cur_frames) + 1) * FRAME_SHIFT_S, 3),
                })
            cur_pieces, cur_frames = [], []
        cur_pieces.append(tstr)
        cur_frames.append(frame)

    if cur_pieces:
        text = "".join(cur_pieces).lstrip(WORD_BOUNDARY)
        if text:
            words.append({
                "word":    text,
                "start_s": round(min(cur_frames)       * FRAME_SHIFT_S, 3),
                "end_s":   round((max(cur_frames) + 1) * FRAME_SHIFT_S, 3),
            })
    return words


def _hyp_to_text(h) -> str:
    if hasattr(h, "text") and h.text:
        return h.text
    y = h.y_sequence
    if isinstance(y, torch.Tensor):
        y = y.tolist()
    elif hasattr(y, "tolist"):
        y = y.tolist()
    return model.tokenizer.ids_to_text(y) if y else ""

In [5]:
# ── Audio loader ──────────────────────────────────────────────────────────────

def load_audio(path: str) -> np.ndarray:
    """Load audio file → int16 PCM at 16 kHz mono."""
    audio, sr = sf.read(path, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != SAMPLE_RATE:
        import librosa
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    return (audio * 32767).clip(-32768, 32767).astype(np.int16)

In [6]:
# ── infer() ───────────────────────────────────────────────────────────────────

def infer(
    audio_int16:   np.ndarray,
    blank_penalty: float = 0.0,
    timestamps:    bool  = False,
) -> Dict:
    """
    Transcribe audio using the current STRATEGY ("ctc" or "tdt").

    blank_penalty:
      CTC  — subtracted from the blank column of the CTC log-prob matrix
              before greedy argmax decode.  Forces more token emissions.
      TDT  — subtracted from the joint output-layer bias at blank_id before
              the TDT beam search, then restored after.  Same bias-patch
              mechanism as model.py _rnnt_blank_penalty().

    Returns:
      {"transcript": str,
       "word_timestamps": [{word, start_s, end_s}, ...]  # [] if timestamps=False
      }
    """
    # ── Preprocess ────────────────────────────────────────────────────────────
    audio_f32 = audio_int16.astype(np.float32) / 32768.0
    sig    = torch.tensor(audio_f32, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    siglen = torch.tensor([len(audio_f32)], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        processed, proc_len = model.preprocessor(input_signal=sig, length=siglen)
        encoded, enc_len    = model.encoder(audio_signal=processed, length=proc_len)

        # ── CTC path ──────────────────────────────────────────────────────────
        if STRATEGY == "ctc":
            # (1, T, vocab+1)
            log_probs = model.ctc_decoder(encoder_output=encoded)

            # Apply blank_penalty before both decoding AND timestamp computation.
            if blank_penalty != 0.0:
                log_probs = log_probs.clone()
                log_probs[:, :, BLANK_ID] -= blank_penalty

            # Decode transcript via NeMo's CTC greedy decoder.
            ctc_result = model.ctc_decoding.ctc_decoder_predictions_tensor(
                log_probs, enc_len.int(), return_hypotheses=True,
            )
            hyp        = ctc_result[0]   # first (only) item in batch
            transcript = _hyp_to_text(hyp)

            # Timestamps from the same blank-penalty-adjusted log_probs.
            word_timestamps: List[Dict] = []
            if timestamps:
                lp_np = log_probs[0].cpu().numpy()   # (T, vocab+1)
                word_timestamps = _compute_ctc_word_timestamps(lp_np)

        # ── TDT path ──────────────────────────────────────────────────────────
        else:
            # Patch joint bias to apply blank_penalty inside TDT beam search.
            # Same mechanism as model.py _rnnt_blank_penalty() context manager.
            _joint_layer.bias.data[BLANK_ID] -= blank_penalty
            try:
                result = model.decoding.rnnt_decoder_predictions_tensor(
                    encoder_output=encoded,
                    encoded_lengths=enc_len,
                    return_hypotheses=True,
                )
            finally:
                # Always restore — even if decode throws.
                _joint_layer.bias.data[BLANK_ID] += blank_penalty

            hyp = result[0][0] if (
                isinstance(result, tuple) and not isinstance(result[0], str)
            ) else result[0]
            transcript = _hyp_to_text(hyp)

            word_timestamps = []
            if timestamps:
                word_timestamps = _hypothesis_to_word_timestamps(hyp)

    return {"transcript": transcript, "word_timestamps": word_timestamps}

In [7]:
# ── Pretty-printer ────────────────────────────────────────────────────────────

def show(label: str, result: Dict) -> None:
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  transcript : {result['transcript']}")
    wts = result.get("word_timestamps", [])
    if wts:
        print(f"  timestamps ({len(wts)} words):")
        for w in wts:
            print(f"    {w['word']:<30} {w['start_s']:>7.3f}s  →  {w['end_s']:>7.3f}s")
    print()

In [8]:
# ── Text cleaning & metrics ───────────────────────────────────────────────────

def clean_text(text: str) -> str:
    text = str(text).strip()
    text = text.replace('\\r\\n', ' ').replace('\\n', ' ').replace('\r\n', ' ').replace('\n', ' ')
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def compute_cer(ref, hyp):
    try:
        r, h = clean_text(ref), clean_text(hyp)
        if len(r) == 0:
            return 0.0 if len(h) == 0 else 1.0
        return cer(r, h)
    except:
        return float("nan")

def compute_wer(ref, hyp):
    try:
        r, h = clean_text(ref), clean_text(hyp)
        if len(r) == 0:
            return 0.0 if len(h) == 0 else 1.0
        return wer(r, h)
    except:
        return float("nan")

def combined_metric(cer_val, wer_val):
    if np.isnan(cer_val) or np.isnan(wer_val):
        return float("nan")
    return ALPHA * cer_val + (1 - ALPHA) * wer_val

In [9]:
# ── VAD: strip leading silence from assistant audio ──────────────────────────

def apply_vad(wav: torch.Tensor, sample_rate: int = SAMPLE_RATE,
               energy_threshold: float = 0.001, frame_ms: int = 20) -> torch.Tensor:
    """
    Remove leading silence from a (1, T) float32 waveform using energy-based VAD.
    Scans frames of `frame_ms` ms; returns audio from the first frame above threshold.
    If the entire signal is silent, the original tensor is returned unchanged.
    """
    frame_size = int(sample_rate * frame_ms / 1000)
    audio = wav.squeeze(0).numpy()
    for start in range(0, len(audio), frame_size):
        frame = audio[start: start + frame_size]
        if float(np.sqrt(np.mean(frame ** 2))) > energy_threshold:
            return wav[:, start:]
    return wav   # fully silent — return as-is


# ── Dataset ───────────────────────────────────────────────────────────────────

class AudioDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            user_wav      = self._load(row["user_audio_path"])
            assistant_wav = self._load(row["assistant_audio_path"])

            # 1. VAD: remove leading silence from assistant
            assistant_wav = apply_vad(assistant_wav)
            user_wav = apply_vad(user_wav)

            # 2. Hard cap on assistant duration
            max_samples = int(MAX_ASSISTANT_DURATION * SAMPLE_RATE)
            if assistant_wav.shape[1] > max_samples:
                assistant_wav = (
                    assistant_wav[:, -max_samples:]
                    if ASSISTANT_TRIM_MODE == "end"
                    else assistant_wav[:, :max_samples]
                )
            
            asst_rms = float(torch.sqrt(torch.mean(assistant_wav ** 2)))
            user_rms = float(torch.sqrt(torch.mean(user_wav ** 2)))
            if asst_rms > 1e-6 and user_rms > 1e-6:
                user_wav = user_wav * (asst_rms / user_rms)

            return {
                "idx":               idx,
                "user_wav":          user_wav,
                "assistant_wav":     assistant_wav,
                "assistant_duration": (assistant_wav.shape[1] / SAMPLE_RATE) * 1000,
                "user_duration": (user_wav.shape[1] / SAMPLE_RATE) * 1000,
                "transcription":     str(row["native_transcription"]),
                "language":          str(row.get("language", "unknown")),
                "user_audio_path":   row["user_audio_path"],
                "ok":                True,
            }
        except Exception as e:
            return {"idx": idx, "ok": False, "error": str(e)}

    def _load(self, path):
        wav, sr = torchaudio.load(path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != SAMPLE_RATE:
            wav = torchaudio.transforms.Resample(sr, SAMPLE_RATE)(wav)
        return wav   # (1, T) float32

def collate_fn(batch):
    return batch


In [10]:
# ── Raw audio features (CPU, once per sample) ─────────────────────────────────
def raw_audio_features(assistant_wav, user_wav, assistant_duration_sec, language):
    user_np      = user_wav.squeeze(0).numpy()
    assistant_np = assistant_wav.squeeze(0).numpy()

    user_rms      = float(np.sqrt(np.mean(user_np**2      + 1e-9)))
    assistant_rms = float(np.sqrt(np.mean(assistant_np**2 + 1e-9)))
    rms_ratio     = user_rms / (assistant_rms + 1e-9)

    user_duration_ms      = user_wav.shape[1] / SAMPLE_RATE * 1000
    assistant_duration_ms = assistant_duration_sec * 1000
    duration_ratio        = user_duration_ms / (assistant_duration_ms + 1e-9)

    frame_size  = int(SAMPLE_RATE * 0.02)
    frames      = [user_np[i:i+frame_size] for i in range(0, len(user_np), frame_size)]
    energies    = np.array([np.sqrt(np.mean(f**2 + 1e-9)) for f in frames])
    transitions = int(np.sum(np.abs(np.diff((energies > 0.01).astype(int)))))
    speaking_rate = transitions / (user_duration_ms / 1000 + 1e-9)

    return {
        "duration_ratio":           duration_ratio,
        "user_rms":                 user_rms,
        "assistant_rms":            assistant_rms,
        "user_assistant_rms_ratio": rms_ratio,
        "speaking_rate":            speaking_rate,
        "language":                 language,
    }

In [11]:
# ── User-only encoder pass (once per sample) ──────────────────────────────────
# Provides H_user_solo and log_probs_solo for context-impact delta features.

@torch.no_grad()
def encode_user_solo(user_wav):
    """
    user_wav : (1, T) float32 tensor
    Returns H_solo (1, T_enc, D) and log_probs_solo (1, T_enc, V+1), both on DEVICE.
    """
    sig    = user_wav.to(DEVICE)
    siglen = torch.tensor([user_wav.shape[1]], device=DEVICE)

    feats, feats_len = model.preprocessor(input_signal=sig, length=siglen)
    H_solo_raw, _    = model.encoder(audio_signal=feats, length=feats_len)
    # encoder returns (B, D, T) — transpose to (B, T, D) for hidden state ops
    H_solo = H_solo_raw.transpose(1, 2)
    log_probs_solo = model.ctc_decoder(encoder_output=H_solo_raw)   # ctc_decoder needs (B, D, T)
    return H_solo, log_probs_solo   # (1,T,D), (1,T,V+1)


# ── Non-fusion baseline ────────────────────────────────────────────────────────

@torch.no_grad()
def transcribe_non_fusion(user_wav):
    """Transcribe user audio alone (no assistant context, no blank_penalty)."""
    sig    = user_wav.to(DEVICE)
    siglen = torch.tensor([user_wav.shape[1]], device=DEVICE)

    feats, feats_len = model.preprocessor(input_signal=sig, length=siglen)
    encoded, enc_len = model.encoder(audio_signal=feats, length=feats_len)

    if STRATEGY == "CTC":
        log_probs        = model.ctc_decoder(encoder_output=encoded)
        decoded = model.ctc_decoding.ctc_decoder_predictions_tensor(
            log_probs, enc_len.int(), return_hypotheses=True,
        )
        hyp = decoded[0]
        return _hyp_to_text(hyp)
    else:  # tdt
        if encoded is None or enc_len is None:
            raise ValueError("encoded/enc_len is None but STRATEGY is tdt")                                                                                                                               

        # capture full joint output via hook to compute mean blank prob                                                                                                                                   
        _joint_outputs = []

        def _hook(module, input, output):                                                                                                                                                                 
            _joint_outputs.append(output.detach())
                                                                                                                                                                                                        
        handle = _joint_layer.register_forward_hook(_hook)
        blank_penalty = 0
        if blank_penalty > 0:
            _joint_layer.bias.data[BLANK_ID] -= blank_penalty
        try:                                                                                                                                                                                              
            result = model.decoding.rnnt_decoder_predictions_tensor(
                encoder_output=encoded,                                                                                                                                                                   
                encoded_lengths=enc_len,
                return_hypotheses=True,
            )
        finally:
            if blank_penalty > 0:
                _joint_layer.bias.data[BLANK_ID] += blank_penalty                                                                                                                                         
            handle.remove()
                                                                                                                                                                                                        
        if _joint_outputs:
            all_logits = torch.cat(_joint_outputs, dim=0)  # (N, ..., V+1)
            dur_logits = all_logits[..., VOCAB_SIZE:]          # duration logits only
            dur_probs = dur_logits.softmax(dim=-1)                                                                                                                                                                
            mean_max_dur_prob = dur_probs[..., -1].mean().item()
            pred_durations = dur_probs.argmax(dim=-1)  # e.g., [0, 1, 3, 2, 4, 4, ...]                                                                                                                                                                                        
            # total frames skipped
            total_frames_skipped = pred_durations.sum().item()                                                                                                                                                                                                                                                                                                                                                    
            # mean frames skipped per step
            mean_frames_skipped = pred_durations.float().mean().item()
        else:                                                                                                                                                                                             
            mean_max_dur_prob = None
                                                                                                                                                                                                        
        hyp = result[0][0] if (
            isinstance(result, tuple) and not isinstance(result[0], str)
        ) else result[0]

        text = _hyp_to_text(hyp)
        if text:                                                                                                                                
            return text, mean_max_dur_prob,mean_frames_skipped
                                                                                                                                                                                                        
        return "", mean_max_dur_prob,mean_frames_skipped

In [12]:
import soundfile as sf
@torch.no_grad()
def build_silence_cache(
    assistant_wav,
    user_wav,
    assistant_duration_sec,
    silence_steps_ms,
    H_user_solo,
    log_probs_solo
):
    cache = {}
    waveforms = []
    user_start_secs = []

    user_duration_s = user_wav.shape[1] / SAMPLE_RATE
    max_combined_samples = int(MAX_COMBINED_DURATION * SAMPLE_RATE)

    for silence_ms in silence_steps_ms:
        s_samples = int(silence_ms / 1000 * SAMPLE_RATE)
        silence = torch.zeros(1, s_samples)

        combined = torch.cat([assistant_wav,silence,user_wav], dim=1)
        sf.write("user_only_2.wav", combined.squeeze(0).numpy(), 16000, format="WAV")
        # combined = torch.cat([user_wav], dim=1)

        if combined.shape[1] > max_combined_samples:
            combined = combined[:, -max_combined_samples:]

        user_start_s = max(
            0.0,
            combined.shape[1] / SAMPLE_RATE - user_duration_s
        )

        waveforms.append(combined)
        user_start_secs.append(user_start_s)

    # -------- batched encoder pass ----------
    lengths = torch.tensor([w.shape[1] for w in waveforms])
    max_len = int(lengths.max().item())

    padded = torch.zeros(len(waveforms), 1, max_len)
    for i, w in enumerate(waveforms):
        padded[i, :, :w.shape[1]] = w

    padded_2d = padded.squeeze(1).to(DEVICE)
    lengths = lengths.to(DEVICE)

    feats, feats_len = model.preprocessor(
        input_signal=padded_2d,
        length=lengths
    )

    H_all, H_lens = model.encoder(
        audio_signal=feats,
        length=feats_len
    )
    # H_all: (B, D, T)

    # -------- batched decoder pass (CTC only) ----------
    if STRATEGY == "ctc":
        log_probs_all = model.ctc_decoder(encoder_output=H_all)  # (B, T, V+1)

    # -------- populate cache ----------
    for i, silence_ms in enumerate(silence_steps_ms):
        if STRATEGY == "ctc":
            cache[silence_ms] = {
                "log_probs_full": log_probs_all[i:i+1],  # (1, T, V+1)
                "encoded":        None,
                "enc_len":        None,
                "user_start_s":   user_start_secs[i],
            }
        else:  # tdt
            cache[silence_ms] = {
                "log_probs_full": None,
                "encoded":        H_all[i:i+1],           # (1, D, T)
                "enc_len":        H_lens[i:i+1],
                "user_start_s":   user_start_secs[i],
            }

    return cache


def decode_with_penalty(                                                      
      log_probs_full,                                                           
      blank_penalty: float,                                                     
      user_start_s: float,                                                      
      encoded=None,                                                             
      enc_len=None,                                                             
  ):                                                                            
      if STRATEGY == "ctc":
          if log_probs_full is None:                                            
              raise ValueError("log_probs_full is None but STRATEGY is ctc")

          lp = log_probs_full.clone()          # (1, T, V+1)                    
          if blank_penalty > 0:
              lp[:, :, BLANK_ID] -= blank_penalty                               
                  
          lp_np = lp[0].cpu().numpy()          # (T, V+1) — matches infer()     
          words = _compute_ctc_word_timestamps(lp_np)
          user_words = [w for w in words if w["start_s"] >= user_start_s]       
          return " ".join(w["word"] for w in user_words)
                                                                                
      else:  # tdt
            if encoded is None or enc_len is None:
                raise ValueError("encoded/enc_len is None but STRATEGY is tdt")                                                                                                                               
    
            # capture full joint output via hook to compute mean blank prob                                                                                                                                   
            _joint_outputs = []

            def _hook(module, input, output):                                                                                                                                                                 
                _joint_outputs.append(output.detach())
                                                                                                                                                                                                                
            handle = _joint_layer.register_forward_hook(_hook)

            if blank_penalty > 0:
                _joint_layer.bias.data[BLANK_ID] -= blank_penalty
            try:                                                                                                                                                                                              
                result = model.decoding.rnnt_decoder_predictions_tensor(
                    encoder_output=encoded,                                                                                                                                                                   
                    encoded_lengths=enc_len,
                    return_hypotheses=True,
                )
            finally:
                if blank_penalty > 0:
                    _joint_layer.bias.data[BLANK_ID] += blank_penalty                                                                                                                                         
                handle.remove()
                
                                                                                                                                                                                                          
            if _joint_outputs:
                all_logits = torch.cat(_joint_outputs, dim=0)  # (N, ..., V+1)
                # all_probs = all_logits.softmax(dim=-1)          # softmax over full vocab                                                                                                                     
                # mean_blank_prob = all_probs[..., BLANK_ID].mean().item()
                dur_logits = all_logits[..., VOCAB_SIZE:]          # duration logits only
                dur_probs = dur_logits.softmax(dim=-1)                                                                                                                                                                
                mean_max_dur_prob = dur_probs[..., -1].mean().item()
                pred_durations = dur_probs.argmax(dim=-1)  # e.g., [0, 1, 3, 2, 4, 4, ...]
                total_frames_skipped = pred_durations.sum().item()                                                                                                                                                    
                mean_frames_skipped = pred_durations.float().mean().item()
            else:                                                                                                                                                                                             
                mean_max_dur_prob = None
                                                                                                                                                                                                                
            hyp = result[0][0] if (
                isinstance(result, tuple) and not isinstance(result[0], str)
            ) else result[0]

            word_timestamps = _hypothesis_to_word_timestamps(hyp)                                                                                                                                             
            if word_timestamps:
                user_words = [w for w in word_timestamps if w["start_s"] >= user_start_s]                                                                                                                     
                text = " ".join(w["word"] for w in user_words)
                return text, mean_max_dur_prob,mean_frames_skipped
                                                                                                                                                                                                                
            return "", mean_max_dur_prob, mean_frames_skipped

In [17]:
# =============================================================
#  fusion_experiment.py
#  TDT feature comparison: user vs fusion at selected frames
#
#  TWO AUDIO CONDITIONS per sample:
#    user    encode/decode user audio alone          (baseline)
#    fusion  encode/decode [assistant|silence|user]  (context)
#
#  SELECTED FRAMES = encoder frame indices where the TDT decoder
#  emits a non-blank token when decoding USER AUDIO ONLY.
#  Both conditions are analysed at these same frame positions.
#
#  ENCODER LEVEL  (at selected frames)
#    user_norm      L2 norm of H_user at fi
#    f_norm         L2 norm of H_fusion at fi + user_offset
#    cos_user_f     cosine similarity between H_user and H_fusion
#    l2_user_f      ||H_fusion[fi+off] - H_user[fi]||_2
#
#  DECODER LEVEL  (joint network logits at emitting steps)
#    blank_prob       P(blank) at each emitting step
#    blank_logit      raw pre-softmax blank logit
#    entropy_top20    Shannon entropy over top-20 non-blank probs
#    logit_gap        top-1 minus top-2 non-blank logit
#    top1_token_prob  max non-blank token probability
#    frames_skipped   TDT predicted duration per step
#    dur_entropy      entropy over TDT duration distribution
#    n_blank_steps    total blank (non-emitting) decoding steps
#    blank_step_ratio n_blank_steps / total_steps
#    mean_blank_run   mean consecutive blanks between emissions
#
#  CLASSIFICATION (combo_delta = user_combined - f_combined):
#    fusion_won   combo_delta > 0   (fusion better)
#    tied         combo_delta == 0
#    degraded     combo_delta < 0   (fusion worse)
# =============================================================
from __future__ import annotations

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from tqdm import tqdm
from torch.utils.data import DataLoader

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
SAMPLE_RATE           = 16000
MAX_COMBINED_DURATION = 15.0
SILENCE_MS            = 1000
BLANK_PENALTY         = 0.0
ALPHA                 = 0.5       # combined_metric = ALPHA*CER + (1-ALPHA)*WER
TOP_K_ENTROPY         = 20    # SentencePiece boundary marker
OUTPUT_CSV            = "fusion_experiment.csv"
NUM_WORKERS           = 2

BLANK_ID     = 8192               # blank token index (last in vocab)
VOCAB_SIZE   = 8193               # token logit count including blank (0-8192)
DEVICE       = torch.device("cuda:0")
_joint_layer = None               # auto-set on first run_experiment() call


# ─────────────────────────────────────────────────────────────────────────────
#  DATA TYPES
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class DecoderFeatures:
    """
    Joint-network features at token-emitting decoding steps.
    Per-step arrays shape (n_emitting,).
    mean_frames_skipped_all covers ALL decoding steps (blank + non-blank).
    """
    n_emitting:              int
    blank_probs:             np.ndarray
    entropy_top20:           np.ndarray
    logit_gap:               np.ndarray
    frames_skipped:          np.ndarray
    mean_frames_skipped_all: float
    top1_token_prob:         np.ndarray
    blank_logit:             np.ndarray
    dur_entropy:             np.ndarray
    n_blank_steps:           int
    blank_step_ratio:        float
    mean_blank_run:          float
    blank_rank:              np.ndarray   # rank of blank among ALL tokens at each emitting step
    blank_to_top_gap:        np.ndarray   # top_token_logit - blank_logit at each emitting step

    def _m(self, arr): return float(np.nanmean(arr)) if self.n_emitting else float("nan")

    @property
    def mean_blank_prob(self):          return self._m(self.blank_probs)
    @property
    def mean_entropy(self):             return self._m(self.entropy_top20)
    @property
    def mean_logit_gap(self):           return self._m(self.logit_gap)
    @property
    def mean_frames_skipped_emit(self): return self._m(self.frames_skipped)
    @property
    def mean_top1_token_prob(self):     return self._m(self.top1_token_prob)
    @property
    def mean_blank_logit(self):         return self._m(self.blank_logit)
    @property
    def mean_dur_entropy(self):         return self._m(self.dur_entropy)
    @property
    def mean_blank_rank(self):          return self._m(self.blank_rank)
    @property
    def mean_blank_to_top_gap(self):    return self._m(self.blank_to_top_gap)

    def to_flat_dict(self, prefix: str) -> dict:
        return {
            f"{prefix}n_emitting":               self.n_emitting,
            f"{prefix}mean_blank_prob":           self.mean_blank_prob,
            f"{prefix}mean_blank_logit":          self.mean_blank_logit,
            f"{prefix}mean_blank_rank":           self.mean_blank_rank,
            f"{prefix}mean_blank_to_top_gap":     self.mean_blank_to_top_gap,
            f"{prefix}mean_entropy_top20":        self.mean_entropy,
            f"{prefix}mean_logit_gap":            self.mean_logit_gap,
            f"{prefix}mean_top1_token_prob":      self.mean_top1_token_prob,
            f"{prefix}mean_frames_skipped_all":   self.mean_frames_skipped_all,
            f"{prefix}mean_frames_skipped_emit":  self.mean_frames_skipped_emit,
            f"{prefix}mean_dur_entropy":          self.mean_dur_entropy,
            f"{prefix}n_blank_steps":             self.n_blank_steps,
            f"{prefix}blank_step_ratio":          self.blank_step_ratio,
            f"{prefix}mean_blank_run":            self.mean_blank_run,
        }


@dataclass
class EncoderFeatures:
    """
    Encoder hidden-state features at selected frame positions.
    Per-frame arrays shape (n_frames,).

      user_norm / f_norm   L2 norm of encoder hidden state
      cos_user_f           cosine similarity between user and fusion states
      l2_user_f            ||H_f[fi+off] - H_u[fi]||  (shift magnitude)
      norm_delta           f_norm - user_norm           (signed energy boost)
      norm_ratio           f_norm / user_norm           (multiplicative boost)
      rel_l2               l2_user_f / user_norm        (shift relative to original)
    """
    n_frames:   int
    user_norm:  np.ndarray
    f_norm:     np.ndarray
    cos_user_f: np.ndarray
    l2_user_f:  np.ndarray
    norm_delta: np.ndarray
    norm_ratio: np.ndarray
    rel_l2:     np.ndarray

    def _m(self, arr): return float(np.nanmean(arr)) if self.n_frames else float("nan")

    def to_flat_dict(self) -> dict:
        return {
            "enc_n_frames":   self.n_frames,
            "enc_user_norm":  self._m(self.user_norm),
            "enc_f_norm":     self._m(self.f_norm),
            "enc_cos_user_f": self._m(self.cos_user_f),
            "enc_l2_user_f":  self._m(self.l2_user_f),
            "enc_norm_delta": self._m(self.norm_delta),
            "enc_norm_ratio": self._m(self.norm_ratio),
            "enc_rel_l2":     self._m(self.rel_l2),
        }


def _empty_decoder(mean_all: float = float("nan")) -> DecoderFeatures:
    z = np.array([])
    return DecoderFeatures(
        n_emitting=0,
        blank_probs=z, entropy_top20=z, logit_gap=z,
        frames_skipped=z, mean_frames_skipped_all=mean_all,
        top1_token_prob=z, blank_logit=z, dur_entropy=z,
        n_blank_steps=0, blank_step_ratio=float("nan"), mean_blank_run=float("nan"),
        blank_rank=z, blank_to_top_gap=z,
    )


def _empty_encoder() -> EncoderFeatures:
    z = np.array([])
    return EncoderFeatures(
        n_frames=0, user_norm=z, f_norm=z,
        cos_user_f=z, l2_user_f=z,
        norm_delta=z, norm_ratio=z, rel_l2=z,
    )


def _frame_suppression_stats(
    user_frames:  List[int],   # emitting encoder frame indices from user decode
    fusion_frames: List[int],  # emitting encoder frame indices from fusion decode (raw, unshifted)
    user_offset:  int,         # encoder frame offset of user speech in fusion audio
    tolerance:    int = 2,     # ±frames for loose matching
) -> dict:
    """
    Decompose the change in n_emitting into suppression events vs new events.

    For each user emitting frame fi, check whether fusion also emits at
    fi + user_offset  (±tolerance).  A match means the frame SURVIVED;
    no match means it was SUPPRESSED (turned blank) by fusion context.

    Fusion frames with no matching user counterpart are NEW emissions —
    tokens the fusion decoder invented that user-only decode did not produce.

    Returns:
      emission_survival_rate   matched / n_user           (1.0 = none suppressed)
      gross_suppression_rate   (n_user - matched) / n_user (0.0 = none suppressed)
      gross_new_rate           n_new / n_user              (0.0 = no hallucinations)
      net_change               f_n_emit - u_n_emit         (< 0 = net suppression)
    """
    n_user = len(user_frames)
    if n_user == 0:
        return dict(emission_survival_rate=float("nan"),
                    gross_suppression_rate=float("nan"),
                    gross_new_rate=float("nan"),
                    net_change=float("nan"))

    # Fusion frames shifted back to user-relative space
    fusion_relative = [f - user_offset for f in fusion_frames if f >= user_offset]
    fusion_set      = set(fusion_relative)

    matched        = 0
    matched_fusion = set()
    for uf in user_frames:
        for d in range(-tolerance, tolerance + 1):
            candidate = uf + d
            if candidate in fusion_set:
                matched += 1
                matched_fusion.add(candidate)
                break

    n_new = len(fusion_set - matched_fusion)

    return dict(
        emission_survival_rate = matched / n_user,
        gross_suppression_rate = (n_user - matched) / n_user,
        gross_new_rate         = n_new / n_user,
        net_change             = len(fusion_relative) - n_user,
    )


@dataclass
class SampleResult:
    idx:         int
    ref:         str
    user_text:   str
    f_text:      str

    user_cer:    float
    user_wer:    float
    f_cer:       float
    f_wer:       float
    combo_delta: float   # user_combined - f_combined; > 0 means fusion helped
    label:       str     # "fusion_won" | "tied" | "degraded"

    user_dec:    DecoderFeatures
    f_dec:       DecoderFeatures
    enc_feats:   EncoderFeatures

    # Frame-level suppression evidence
    emission_survival_rate:  float   # fraction of user emitting frames that survive in fusion
    gross_suppression_rate:  float   # fraction of user frames turned blank by fusion
    gross_new_rate:          float   # fusion-only emissions / n_user (hallucination rate)
    net_change:              float   # f_n_emit - u_n_emit

    silence_ms:    int
    blank_penalty: float
    raw_feats:     dict = field(default_factory=dict)

    def to_flat_dict(self) -> dict:
        return {
            "idx":                    self.idx,
            "ref":                    self.ref,
            "user_text":              self.user_text,
            "f_text":                 self.f_text,
            "user_cer":               self.user_cer,
            "user_wer":               self.user_wer,
            "f_cer":                  self.f_cer,
            "f_wer":                  self.f_wer,
            "combo_delta":            self.combo_delta,
            "label":                  self.label,
            "emission_survival_rate": self.emission_survival_rate,
            "gross_suppression_rate": self.gross_suppression_rate,
            "gross_new_rate":         self.gross_new_rate,
            "net_change":             self.net_change,
            "silence_ms":             self.silence_ms,
            "blank_penalty":          self.blank_penalty,
            **self.user_dec.to_flat_dict("user_dec_"),
            **self.f_dec.to_flat_dict("f_dec_"),
            **self.enc_feats.to_flat_dict(),
            **self.raw_feats,
        }


# ─────────────────────────────────────────────────────────────────────────────
#  HYPOTHESIS HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _hypothesis_to_word_timestamps(hypothesis) -> List[Dict]:
    y_seq = hypothesis.y_sequence
    if isinstance(y_seq, torch.Tensor):  y_seq = y_seq.tolist()
    elif hasattr(y_seq, "tolist"):       y_seq = y_seq.tolist()

    tss = (hypothesis.timestep if hasattr(hypothesis, "timestep")
           else getattr(hypothesis, "timestamp", None))
    if isinstance(tss, dict):         tss = tss.get("timestep", [])
    if isinstance(tss, torch.Tensor): tss = tss.tolist()
    elif hasattr(tss, "tolist"):      tss = tss.tolist()

    if not y_seq or not tss or len(y_seq) != len(tss):
        return []

    token_strs = model.tokenizer.ids_to_tokens(list(y_seq))
    words: List[Dict] = []
    cur_pieces: List[str] = []
    cur_frames: List[int] = []

    for tstr, frame in zip(token_strs, tss):
        frame = int(frame)
        if tstr.startswith(WORD_BOUNDARY) and cur_pieces:
            text = "".join(cur_pieces).lstrip(WORD_BOUNDARY)
            if text:
                words.append({
                    "word":    text,
                    "start_s": round(min(cur_frames) * FRAME_SHIFT_S, 3),
                    "end_s":   round((max(cur_frames) + 1) * FRAME_SHIFT_S, 3),
                })
            cur_pieces, cur_frames = [], []
        cur_pieces.append(tstr)
        cur_frames.append(frame)

    if cur_pieces:
        text = "".join(cur_pieces).lstrip(WORD_BOUNDARY)
        if text:
            words.append({
                "word":    text,
                "start_s": round(min(cur_frames) * FRAME_SHIFT_S, 3),
                "end_s":   round((max(cur_frames) + 1) * FRAME_SHIFT_S, 3),
            })
    return words


def _hyp_to_text(h) -> str:
    if hasattr(h, "text") and h.text:
        return h.text
    y = h.y_sequence
    if isinstance(y, torch.Tensor): y = y.tolist()
    elif hasattr(y, "tolist"):      y = y.tolist()
    return model.tokenizer.ids_to_text(y) if y else ""


def _extract_hyp(result):
    return (
        result[0][0]
        if isinstance(result, tuple) and not isinstance(result[0], str)
        else result[0]
    )


def _hyp_token_frames(hyp) -> List[int]:
    tss = (hyp.timestep if hasattr(hyp, "timestep")
           else getattr(hyp, "timestamp", None))
    if isinstance(tss, dict):         tss = tss.get("timestep", [])
    if isinstance(tss, torch.Tensor): return tss.tolist()
    if hasattr(tss, "tolist"):        return tss.tolist()
    return list(tss) if tss else []


# ─────────────────────────────────────────────────────────────────────────────
#  JOINT-NETWORK HOOK
# ─────────────────────────────────────────────────────────────────────────────

class _JointHook:
    """
    Forward hook on _joint_layer.
    TDT layout: [..., :VOCAB_SIZE] = token logits, [..., VOCAB_SIZE:] = duration logits
    """
    def __init__(self):
        self._outputs: List[torch.Tensor] = []
        self._handle = None

    def __enter__(self):
        def _hook(_, __, out): self._outputs.append(out.detach())
        self._handle = _joint_layer.register_forward_hook(_hook)
        return self

    def __exit__(self, *_):
        if self._handle: self._handle.remove()

    def collect(self) -> Tuple[torch.Tensor, torch.Tensor]:
        if not self._outputs:
            return torch.zeros(0, VOCAB_SIZE), torch.zeros(0, 0)
        flat = torch.cat([o.reshape(-1, o.shape[-1]) for o in self._outputs], dim=0)
        return flat[:, :VOCAB_SIZE], flat[:, VOCAB_SIZE:]


# ─────────────────────────────────────────────────────────────────────────────
#  DECODE
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def _decode_tdt_raw(
    encoded:       torch.Tensor,
    enc_len:       torch.Tensor,
    blank_penalty: float = BLANK_PENALTY,
) -> Tuple[torch.Tensor, torch.Tensor, List[int], object]:
    """Returns (token_logits, dur_logits, token_frames, hyp)."""
    with _JointHook() as hook:
        if blank_penalty > 0:
            _joint_layer.bias.data[BLANK_ID] -= blank_penalty
        try:
            result = model.decoding.rnnt_decoder_predictions_tensor(
                encoder_output=encoded,
                encoded_lengths=enc_len.to(encoded.device).int(),
                return_hypotheses=True,
            )
        finally:
            if blank_penalty > 0:
                _joint_layer.bias.data[BLANK_ID] += blank_penalty

    tok_l, dur_l = hook.collect()
    hyp          = _extract_hyp(result)
    return tok_l, dur_l, _hyp_token_frames(hyp), hyp


# ─────────────────────────────────────────────────────────────────────────────
#  DECODER FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def _blank_run_lengths(emit_mask: torch.Tensor) -> List[int]:
    runs, count = [], 0
    for is_emit in emit_mask.tolist():
        if not is_emit:
            count += 1
        else:
            runs.append(count)
            count = 0
    runs.append(count)
    return runs


def _compute_decoder_features(
    token_logits: torch.Tensor,
    dur_logits:   torch.Tensor,
    token_frames: List[int],
    user_start_s: float = 0.0,
    top_k:        int   = TOP_K_ENTROPY,
) -> DecoderFeatures:
    """
    user_start_s=0.0  → user condition, all emitting steps used.
    user_start_s>0    → fusion condition, only steps with frame >= user_start_frame.
    """
    N_all = token_logits.shape[0]

    if dur_logits.shape[-1] > 0:
        mean_all = dur_logits.softmax(dim=-1).argmax(dim=-1).float().mean().item()
    else:
        mean_all = float("nan")

    emit_mask_all = token_logits.argmax(dim=-1) != BLANK_ID
    n_blank_all   = int((~emit_mask_all).sum().item())
    blank_ratio   = n_blank_all / N_all if N_all > 0 else float("nan")

    emit_tok = token_logits[emit_mask_all]
    emit_dur = dur_logits[emit_mask_all]

    emit_mask_filtered = None
    if user_start_s > 0.0 and token_frames:
        user_start_frame = user_start_s / FRAME_SHIFT_S
        umask = torch.tensor(
            [f >= user_start_frame for f in token_frames],
            dtype=torch.bool, device=emit_tok.device,
        )
        if umask.shape[0] == emit_tok.shape[0]:
            emit_tok           = emit_tok[umask]
            emit_dur           = emit_dur[umask]
            emit_mask_filtered = umask

    E = emit_tok.shape[0]
    if E == 0:
        return _empty_decoder(mean_all)

    if emit_mask_filtered is not None:
        filtered_frames = sorted([f for f in token_frames
                                   if f >= user_start_s / FRAME_SHIFT_S])
        if len(filtered_frames) > 1:
            gaps = [filtered_frames[i+1] - filtered_frames[i] - 1
                    for i in range(len(filtered_frames) - 1)]
            mean_blank_run = float(np.mean(gaps)) if gaps else 0.0
        else:
            mean_blank_run = 0.0
    else:
        runs = _blank_run_lengths(emit_mask_all)
        mean_blank_run = float(np.mean(runs)) if runs else 0.0

    emit_probs = emit_tok.softmax(dim=-1)
    blank_probs = emit_probs[:, BLANK_ID].cpu().numpy()
    blank_logit_vals = emit_tok[:, BLANK_ID]                            # (E,) tensor

    keep      = [i for i in range(emit_tok.shape[-1]) if i != BLANK_ID]
    nb        = emit_tok[:, keep]
    k         = min(top_k, nb.shape[-1])
    topk_l, _ = torch.topk(nb, k, dim=-1)
    topk_p    = topk_l.softmax(dim=-1)
    entropy   = -(topk_p * topk_p.clamp(min=1e-9).log()).sum(dim=-1)
    entropy_np = entropy.cpu().numpy()
    gap = (topk_l[:, 0] - topk_l[:, 1]).cpu().numpy() if k >= 2 else np.zeros(E, dtype=np.float32)
    top1_prob = topk_p[:, 0].cpu().numpy()

    # ── Blank rank: how many tokens have logit > blank at each emitting step ──
    # rank 1 = top logit; at emitting steps blank is always rank >= 2
    n_above_blank = (emit_tok > blank_logit_vals.unsqueeze(1)).sum(dim=-1)  # (E,)
    blank_rank_np = (n_above_blank + 1).float().cpu().numpy()               # (E,)

    # ── Blank-to-top gap: top_token_logit - blank_logit ──────────────────────
    # always >= 0 at emitting steps (top token beat blank)
    # if blank were top the gap would be 0, but that can't happen here
    top_logit = emit_tok.max(dim=-1).values                                 # (E,)
    blank_to_top_gap_np = (top_logit - blank_logit_vals).cpu().numpy()      # (E,)

    if emit_dur.shape[-1] > 0:
        dur_p          = emit_dur.softmax(dim=-1)
        frames_skipped = dur_p.argmax(dim=-1).float().cpu().numpy()
        dur_entropy_np = -(dur_p * dur_p.clamp(min=1e-9).log()).sum(dim=-1).cpu().numpy()
    else:
        frames_skipped = np.full(E, float("nan"))
        dur_entropy_np = np.full(E, float("nan"))

    return DecoderFeatures(
        n_emitting=E,
        blank_probs=blank_probs, entropy_top20=entropy_np,
        logit_gap=gap, frames_skipped=frames_skipped,
        mean_frames_skipped_all=mean_all,
        top1_token_prob=top1_prob, blank_logit=blank_logit_vals.cpu().numpy(),
        dur_entropy=dur_entropy_np,
        n_blank_steps=n_blank_all, blank_step_ratio=blank_ratio,
        mean_blank_run=mean_blank_run,
        blank_rank=blank_rank_np,
        blank_to_top_gap=blank_to_top_gap_np,
    )


# ─────────────────────────────────────────────────────────────────────────────
#  ENCODER FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────

def _compute_encoder_features(
    H_user:          torch.Tensor,   # (1, D, T_user)
    H_f:             torch.Tensor,   # (1, D, T_fusion)
    selected_frames: List[int],
    user_offset:     int,
) -> EncoderFeatures:
    """
    At each selected frame fi (from user-only decode):
      H_user[fi]             — user-only encoder state
      H_f[fi + user_offset]  — fusion encoder state at same user-speech position
    """
    h_u = H_user[0].T   # (T_user, D)
    h_f = H_f[0].T      # (T_f,    D)
    T_u, T_f = h_u.shape[0], h_f.shape[0]

    u_norms, f_norms, cos_sims, l2_dists = [], [], [], []
    n_deltas, n_ratios, rel_l2s = [], [], []

    for fi in selected_frames:
        fi_f = fi + user_offset
        if fi >= T_u or fi_f >= T_f:
            continue
        v_u  = h_u[fi]
        v_f  = h_f[fi_f]
        n_u  = v_u.norm().item()
        n_f  = v_f.norm().item()
        l2   = (v_f - v_u).norm().item()

        u_norms.append(n_u)
        f_norms.append(n_f)
        cos_sims.append(F.cosine_similarity(v_u.unsqueeze(0), v_f.unsqueeze(0)).item())
        l2_dists.append(l2)
        n_deltas.append(n_f - n_u)
        n_ratios.append(n_f / n_u if n_u > 0 else float("nan"))
        rel_l2s.append(l2 / n_u  if n_u > 0 else float("nan"))

    n = len(u_norms)
    if n == 0:
        return _empty_encoder()

    a = lambda lst: np.array(lst, dtype=np.float32)
    return EncoderFeatures(
        n_frames=n,
        user_norm=a(u_norms),  f_norm=a(f_norms),
        cos_user_f=a(cos_sims), l2_user_f=a(l2_dists),
        norm_delta=a(n_deltas), norm_ratio=a(n_ratios),
        rel_l2=a(rel_l2s),
    )


# ─────────────────────────────────────────────────────────────────────────────
#  AUDIO UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def encode_audio(wav: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """wav : (1, T) → (H (1,D,T_enc), enc_len (1,) int on DEVICE)"""
    sig    = wav.to(DEVICE)
    siglen = torch.tensor([wav.shape[1]], device=DEVICE)
    feats, feats_len = model.preprocessor(input_signal=sig, length=siglen)
    H, enc_len       = model.encoder(audio_signal=feats, length=feats_len)
    return H.to(DEVICE), enc_len.to(DEVICE).int()


def build_fusion_audio(
    assistant_wav: torch.Tensor,
    user_wav:      torch.Tensor,
    silence_ms:    int = SILENCE_MS,
) -> Tuple[torch.Tensor, float]:
    """[assistant | silence | user] → (combined, user_start_s)"""
    silence  = torch.zeros(1, int(silence_ms / 1000 * SAMPLE_RATE))
    combined = torch.cat([assistant_wav, silence, user_wav], dim=1)
    max_samp = int(MAX_COMBINED_DURATION * SAMPLE_RATE)
    if combined.shape[1] > max_samp:
        combined = combined[:, -max_samp:]
    user_start_s = max(
        0.0, combined.shape[1] / SAMPLE_RATE - user_wav.shape[1] / SAMPLE_RATE
    )
    return combined, user_start_s


# ─────────────────────────────────────────────────────────────────────────────
#  CLASSIFICATION
# ─────────────────────────────────────────────────────────────────────────────

def combined_metric(cer: float, wer: float) -> float:
    return ALPHA * cer + (1 - ALPHA) * wer


def classify(combo_delta: float) -> str:
    if np.isnan(combo_delta): return "error"
    if combo_delta > 0:       return "fusion_won"
    if combo_delta == 0:      return "tied"
    return "degraded"


# ─────────────────────────────────────────────────────────────────────────────
#  PER-SAMPLE PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_sample(
    idx:                int,
    ref:                str,
    user_wav:           torch.Tensor,
    assistant_wav:      torch.Tensor,
    assistant_duration: float,
    user_duration:      float,
    language:           str,
    silence_ms:         int   = SILENCE_MS,
    blank_penalty:      float = BLANK_PENALTY,
) -> Optional[SampleResult]:
    # ── Step 1: User-only ────────────────────────────────────────────────────
    try:
        H_user, enc_len_u = encode_audio(user_wav)
        u_tok, u_dur, selected_frames, u_hyp = _decode_tdt_raw(H_user, enc_len_u, blank_penalty)
        user_words = _hypothesis_to_word_timestamps(u_hyp)
        user_text  = " ".join(w["word"] for w in user_words)
        user_dec   = _compute_decoder_features(u_tok, u_dur, selected_frames, user_start_s=0.0)
        user_cer   = compute_cer(ref, user_text)
        user_wer   = compute_wer(ref, user_text)
        user_comb  = combined_metric(user_cer, user_wer)
    except Exception as e:
        print(f"[{idx}] user decode failed: {e}")
        return None

    # ── Step 2: Fusion ───────────────────────────────────────────────────────
    try:
        f_wav, user_start_f = build_fusion_audio(assistant_wav, user_wav, silence_ms)
        H_f, enc_len_f      = encode_audio(f_wav)
        f_tok, f_dur, f_frames, f_hyp = _decode_tdt_raw(H_f, enc_len_f, blank_penalty)
        f_words  = _hypothesis_to_word_timestamps(f_hyp)
        f_text   = " ".join(w["word"] for w in f_words if w["start_s"] >= user_start_f)
        f_dec    = _compute_decoder_features(f_tok, f_dur, f_frames, user_start_s=user_start_f)
        f_cer    = compute_cer(ref, f_text)
        f_wer    = compute_wer(ref, f_text)
        f_comb   = combined_metric(f_cer, f_wer)
    except Exception as e:
        print(f"[{idx}] fusion decode failed: {e}")
        return None

    # ── Step 3: Encoder analysis + suppression stats ─────────────────────────
    user_offset  = int(user_start_f / FRAME_SHIFT_S)
    enc_feats    = _compute_encoder_features(H_user, H_f, selected_frames, user_offset)
    supp_stats   = _frame_suppression_stats(selected_frames, f_frames, user_offset)

    combo_delta = user_comb - f_comb
    raw_feats   = raw_audio_features(assistant_wav, user_wav, assistant_duration, language)

    return SampleResult(
        idx=idx, ref=ref,
        user_text=user_text, f_text=f_text,
        user_cer=user_cer, user_wer=user_wer,
        f_cer=f_cer,       f_wer=f_wer,
        combo_delta=combo_delta,
        label=classify(combo_delta),
        user_dec=user_dec, f_dec=f_dec,
        enc_feats=enc_feats,
        emission_survival_rate = supp_stats["emission_survival_rate"],
        gross_suppression_rate = supp_stats["gross_suppression_rate"],
        gross_new_rate         = supp_stats["gross_new_rate"],
        net_change             = supp_stats["net_change"],
        silence_ms=silence_ms, blank_penalty=blank_penalty,
        raw_feats=raw_feats,
    )


# ─────────────────────────────────────────────────────────────────────────────
#  MAIN LOOP
# ─────────────────────────────────────────────────────────────────────────────

def run_experiment(
    df:            pd.DataFrame,
    silence_ms:    int   = SILENCE_MS,
    blank_penalty: float = BLANK_PENALTY,
    output_csv:    str   = OUTPUT_CSV,
) -> pd.DataFrame:
    global _joint_layer
    if _joint_layer is None:
        _joint_layer = model.joint.joint_net[2]

    dataset    = AudioDataset(df)
    dataloader = DataLoader(
        dataset, batch_size=1, num_workers=NUM_WORKERS,
        collate_fn=collate_fn, prefetch_factor=2, pin_memory=True,
    )

    records = []
    for batch in tqdm(dataloader, desc="Experiment"):
        item = batch[0]
        if not item["ok"]:
            print(f"[{item['idx']}] load failed: {item.get('error')}")
            continue

        result = run_sample(
            idx=item["idx"], ref=item["transcription"],
            user_wav=item["user_wav"], assistant_wav=item["assistant_wav"],
            assistant_duration=item["assistant_duration"],
            user_duration=item["user_duration"],
            language=item["language"],
            silence_ms=silence_ms, blank_penalty=blank_penalty,
        )
        if result is not None:
            records.append(result.to_flat_dict())

        torch.cuda.empty_cache()

    results_df = pd.DataFrame(records)
    results_df.to_csv(output_csv, index=False)
    print(f"\nSaved {output_csv}  ({len(results_df)} rows)")
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
#  ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

_DEC_STATS = [
    "n_emitting",
    "mean_blank_prob", "mean_blank_logit",
    "mean_blank_rank", "mean_blank_to_top_gap",
    "mean_entropy_top20", "mean_logit_gap", "mean_top1_token_prob",
    "mean_frames_skipped_all", "mean_frames_skipped_emit",
    "mean_dur_entropy", "n_blank_steps", "blank_step_ratio", "mean_blank_run",
]
_USER_DEC_COLS = [f"user_dec_{s}" for s in _DEC_STATS]
_F_DEC_COLS    = [f"f_dec_{s}"    for s in _DEC_STATS]
_ENC_COLS      = ["enc_n_frames", "enc_user_norm", "enc_f_norm",
                  "enc_cos_user_f", "enc_l2_user_f",
                  "enc_norm_delta", "enc_norm_ratio", "enc_rel_l2"]
_SUPP_COLS     = ["emission_survival_rate", "gross_suppression_rate",
                  "gross_new_rate", "net_change"]
_METRIC_COLS   = ["user_cer", "user_wer", "f_cer", "f_wer", "combo_delta"]
ALL_FEAT_COLS  = _USER_DEC_COLS + _F_DEC_COLS + _ENC_COLS + _SUPP_COLS


def analyze(results_df: pd.DataFrame) -> pd.DataFrame:
    total = len(results_df)

    print("\n===== LABEL DISTRIBUTION =====")
    for lbl, cnt in results_df["label"].value_counts().items():
        print(f"  {lbl:<15s}: {cnt:5d}  ({cnt / total * 100:.1f}%)")

    def _section(title, cols):
        avail = [c for c in cols if c in results_df.columns]
        if avail:
            print(f"\n===== {title} =====")
            print(results_df.groupby("label")[avail].mean().T.to_string())

    _section("DECODER FEATURES — user",          _USER_DEC_COLS)
    _section("DECODER FEATURES — fusion",        _F_DEC_COLS)
    _section("ENCODER FEATURES",                 _ENC_COLS)
    _section("FRAME SUPPRESSION EVIDENCE",       _SUPP_COLS)
    _section("METRICS",                          _METRIC_COLS)

    avail = [c for c in ALL_FEAT_COLS if c in results_df.columns]
    return results_df.groupby("label")[avail].mean()


def compare_classes(
    results_df: pd.DataFrame,
    class_a:    str,
    class_b:    str,
) -> pd.DataFrame:
    a     = results_df[results_df["label"] == class_a]
    b     = results_df[results_df["label"] == class_b]
    avail = [c for c in ALL_FEAT_COLS if c in results_df.columns]
    rows  = []
    for col in avail:
        ma, mb = a[col].mean(), b[col].mean()
        rows.append({"feature": col, class_a: ma, class_b: mb, "delta(a-b)": ma - mb})
    return pd.DataFrame(rows).set_index("feature")


# ─────────────────────────────────────────────────────────────────────────────
#  ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = pd.read_csv("final_fixed_clean.csv")
    print(f"Total rows: {len(df)}")

    results_df = run_experiment(df)
    analyze(results_df)

    print("\n===== fusion_won vs degraded =====")
    print(compare_classes(results_df, "fusion_won", "degraded").to_string())

Total rows: 283


Experiment:   0%|          | 0/283 [00:00<?, ?it/s]

Experiment:  83%|████████▎ | 235/283 [00:50<00:07,  6.29it/s]

[233] load failed: Failed to create AudioDecoder for /home/ubuntu/Pavan/acoustic_fusion/filtered_audio_pairs/agent_9d73eead-b7e8-4b3c-a5dc-8e32bfbbb54e__63d446cd-6867-426c-8bf6-66ee2c66f495__pair024__user.wav: Could not open input file: /home/ubuntu/Pavan/acoustic_fusion/filtered_audio_pairs/agent_9d73eead-b7e8-4b3c-a5dc-8e32bfbbb54e__63d446cd-6867-426c-8bf6-66ee2c66f495__pair024__user.wav No such file or directory


Experiment: 100%|██████████| 283/283 [01:00<00:00,  4.70it/s]


Saved fusion_experiment.csv  (282 rows)

===== LABEL DISTRIBUTION =====
  tied           :   128  (45.4%)
  degraded       :   101  (35.8%)
  fusion_won     :    53  (18.8%)

===== DECODER FEATURES — user =====
label                               degraded  fusion_won       tied
user_dec_n_emitting                 7.811881    8.490566   7.914062
user_dec_mean_blank_prob            0.046640    0.060938   0.034125
user_dec_mean_blank_logit         -27.989286  -28.258800 -30.172260
user_dec_mean_blank_rank            4.917509    4.970946   5.058687
user_dec_mean_blank_to_top_gap      7.316476    6.801393   8.730333
user_dec_mean_entropy_top20         0.342340    0.502732   0.191368
user_dec_mean_logit_gap             5.656330    5.000173   6.727666
user_dec_mean_top1_token_prob       0.895794    0.837204   0.938979
user_dec_mean_frames_skipped_all    2.694251    2.630273   2.893792
user_dec_mean_frames_skipped_emit   2.560156    2.454286   2.774691
user_dec_mean_dur_entropy           0.93

In [20]:
# =============================================================
#  listen_degraded.py
#  Load the most-degraded fusion samples, build their combined
#  [assistant | silence | user] audio, and save all three
#  versions (user only, assistant only, fusion combined) as
#  WAVs so you can listen and understand why fusion hurt them.
#
#  Depends only on: fusion_experiment.csv + final_fixed.csv
#  No model needed.
# =============================================================
import os
import soundfile as sf
import numpy as np
import pandas as pd
import torch
import torchaudio
from pathlib import Path

# ─────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────
RESULTS_CSV   = "fusion_experiment.csv"     # from run_experiment()
SOURCE_CSV    = "final_fixed_clean.csv"           # original dataset CSV
OUTPUT_DIR    = "degraded_audio_samples"    # folder to save WAVs
TOP_N         = 30                          # how many worst samples to save
SAMPLE_RATE   = 16000
SILENCE_MS    = 1000                        # must match SILENCE_MS used in experiment

# Column names in final_fixed.csv that hold audio file paths.
# Adjust these if your CSV uses different column names.
USER_PATH_COL      = "user_audio_path"
ASSISTANT_PATH_COL = "assistant_audio_path"

# ─────────────────────────────────────────────────────────────
#  LOAD AND SORT RESULTS
# ─────────────────────────────────────────────────────────────
results_df = pd.read_csv(RESULTS_CSV)
source_df  = pd.read_csv(SOURCE_CSV)

# Print columns so you can verify path column names
print("fusion_experiment.csv columns:", list(results_df.columns[:20]))
print("final_fixed.csv columns:",       list(source_df.columns[:20]))

degraded_df = (
    results_df[results_df["label"] == "degraded"]
    .sort_values("combo_delta")          # most negative first
    .head(TOP_N)
    .reset_index(drop=True)
)

print(f"\nTop {TOP_N} most degraded samples:")
print(degraded_df[["idx", "ref", "user_text", "f_text",
                    "user_cer", "f_cer", "combo_delta"]].to_string())

# ─────────────────────────────────────────────────────────────
#  HELPERS
# ─────────────────────────────────────────────────────────────

def load_wav(path: str) -> torch.Tensor:
    """Load audio file → (1, T) float32 tensor at SAMPLE_RATE."""
    wav, sr = torchaudio.load(path)
    if sr != SAMPLE_RATE:
        wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)
    return wav


def build_fusion_audio(
    assistant_wav: torch.Tensor,
    user_wav:      torch.Tensor,
    silence_ms:    int = SILENCE_MS,
) -> torch.Tensor:
    silence  = torch.zeros(1, int(silence_ms / 1000 * SAMPLE_RATE))
    return torch.cat([assistant_wav, silence, user_wav], dim=1)


def save_wav(wav: torch.Tensor, path: str):
    sf.write(path, wav.squeeze(0).numpy(), SAMPLE_RATE, format="WAV")
    print(f"  saved → {path}")


# ─────────────────────────────────────────────────────────────
#  RESOLVE AUDIO PATHS
# ─────────────────────────────────────────────────────────────
# Try to find path columns — check results CSV first, then source CSV.
def resolve_paths(row_result, source_df):
    idx = int(row_result["idx"])

    # Try getting paths directly from fusion_experiment.csv
    user_path = row_result.get(USER_PATH_COL, None)
    asst_path = row_result.get(ASSISTANT_PATH_COL, None)

    # Fall back to final_fixed.csv if not found in results
    if (pd.isna(user_path) or not user_path) and idx < len(source_df):
        src_row   = source_df.iloc[idx]
        user_path = src_row.get(USER_PATH_COL, src_row.get("user_path", None))
        asst_path = src_row.get(ASSISTANT_PATH_COL, src_row.get("assistant_path",
                    src_row.get("agent_audio_path", None)))

    return str(user_path) if user_path else None, \
           str(asst_path) if asst_path else None


# ─────────────────────────────────────────────────────────────
#  BUILD AND SAVE
# ─────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

for rank, (_, row) in enumerate(degraded_df.iterrows(), start=1):
    idx         = int(row["idx"])
    combo_delta = row["combo_delta"]
    user_cer    = row["user_cer"]
    f_cer       = row["f_cer"]
    ref         = row["ref"]
    user_text   = row["user_text"]
    f_text      = row["f_text"]

    user_path, asst_path = resolve_paths(row, source_df)

    print(f"\n[{rank}] idx={idx}  combo_delta={combo_delta:.3f}  "
          f"user_cer={user_cer:.3f} → f_cer={f_cer:.3f}")
    print(f"  ref       : {ref}")
    print(f"  user_text : {user_text}")
    print(f"  f_text    : {f_text}")

    if not user_path or not os.path.exists(user_path):
        print(f"  [SKIP] user audio not found: {user_path}")
        continue
    if not asst_path or not os.path.exists(asst_path):
        print(f"  [SKIP] assistant audio not found: {asst_path}")
        continue

    user_wav = load_wav(user_path)
    asst_wav = load_wav(asst_path)
    fuse_wav = build_fusion_audio(asst_wav, user_wav, SILENCE_MS)

    prefix = f"{OUTPUT_DIR}/rank{rank:02d}_idx{idx}_delta{combo_delta:.3f}"
    save_wav(user_wav, f"{prefix}__user_only.wav")
    save_wav(asst_wav, f"{prefix}__assistant_only.wav")
    save_wav(fuse_wav, f"{prefix}__fusion_combined.wav")

    # Write a text label file alongside so you know what each audio says
    with open(f"{prefix}__info.txt", "w") as f:
        f.write(f"idx:            {idx}\n")
        f.write(f"combo_delta:    {combo_delta:.4f}\n")
        f.write(f"user_cer:       {user_cer:.4f}   user_wer: {row['user_wer']:.4f}\n")
        f.write(f"f_cer:          {f_cer:.4f}   f_wer:    {row['f_wer']:.4f}\n")
        f.write(f"\nREFERENCE:      {ref}\n")
        f.write(f"USER DECODE:    {user_text}\n")
        f.write(f"FUSION DECODE:  {f_text}\n")
        f.write(f"\nuser audio:     {user_path}\n")
        f.write(f"assistant audio:{asst_path}\n")

print(f"\nDone. Audio saved to: {OUTPUT_DIR}/")
print("Files per sample:")
print("  *__user_only.wav       — user speech alone (model decoded this correctly)")
print("  *__assistant_only.wav  — assistant speech that was prepended")
print("  *__fusion_combined.wav — [assistant | 1s silence | user] (model decoded this badly)")

fusion_experiment.csv columns: ['idx', 'ref', 'user_text', 'f_text', 'user_cer', 'user_wer', 'f_cer', 'f_wer', 'combo_delta', 'label', 'emission_survival_rate', 'gross_suppression_rate', 'gross_new_rate', 'net_change', 'silence_ms', 'blank_penalty', 'user_dec_n_emitting', 'user_dec_mean_blank_prob', 'user_dec_mean_blank_logit', 'user_dec_mean_blank_rank']
final_fixed.csv columns: ['annotation_id', 'annotator', 'assistant_audio_path', 'background_voice', 'created_at', 'id', 'language', 'lead_time', 'native_transcription', 'noise_level', 'noise_severity', 'transcription', 'updated_at', 'user_audio_path', 'user_text', 'bot_cleanliness']

Top 30 most degraded samples:
    idx                                                      ref                                        user_text                                    f_text  user_cer     f_cer  combo_delta
0    22                            Doctor Elizabath Jacob.\n\n\n                          doctor elizabeth jacker                         

  saved → degraded_audio_samples/rank06_idx113_delta-0.417__user_only.wav
  saved → degraded_audio_samples/rank06_idx113_delta-0.417__assistant_only.wav
  saved → degraded_audio_samples/rank06_idx113_delta-0.417__fusion_combined.wav

[7] idx=247  combo_delta=-0.414  user_cer=0.214 → f_cer=0.643
  ref       : अभी क्यों जाना चाहते हैं 
  user_text : अभी क्यों जानना चाहते
  f_text    : जानना चाहते
  saved → degraded_audio_samples/rank07_idx247_delta-0.414__user_only.wav
  saved → degraded_audio_samples/rank07_idx247_delta-0.414__assistant_only.wav
  saved → degraded_audio_samples/rank07_idx247_delta-0.414__fusion_combined.wav

[8] idx=198  combo_delta=-0.414  user_cer=0.143 → f_cer=0.571
  ref       : नहीं हैं  पसंद  नहीं आया 
  user_text : नहीं आप पसंद नहीं आया
  f_text    : नहीं आप push नहीं हैं
  saved → degraded_audio_samples/rank08_idx198_delta-0.414__user_only.wav
  saved → degraded_audio_samples/rank08_idx198_delta-0.414__assistant_only.wav
  saved → degraded_audio_samples/rank08_i

In [19]:
import pandas as pd

RESULTS_CSV  = "fusion_experiment.csv"
SOURCE_CSV   = "final_fixed.csv"
CLEAN_CSV    = "final_fixed_clean_2.csv"      # original minus the top-N degraded
DEGRADED_CSV = "final_fixed_degraded_2.csv"   # the top-N degraded rows only
TOP_N        = 10

# ── Load ──────────────────────────────────────────────────────
results_df = pd.read_csv(RESULTS_CSV)
source_df  = pd.read_csv(SOURCE_CSV)

# ── Get top-N degraded idx values ─────────────────────────────
degraded_idx = (
    results_df[results_df["label"] == "degraded"]
    .sort_values("combo_delta")
    .head(TOP_N)["idx"]
    .astype(int)
    .tolist()
)

print(f"Removing {len(degraded_idx)} rows with idx: {degraded_idx}")

# ── Split ─────────────────────────────────────────────────────
degraded_rows = source_df[source_df.index.isin(degraded_idx)]
clean_rows    = source_df[~source_df.index.isin(degraded_idx)]

# ── Save ──────────────────────────────────────────────────────
clean_rows.to_csv(CLEAN_CSV,    index=False)
degraded_rows.to_csv(DEGRADED_CSV, index=False)

print(f"original rows : {len(source_df)}")
print(f"clean rows    : {len(clean_rows)}  → {CLEAN_CSV}")
print(f"degraded rows : {len(degraded_rows)}  → {DEGRADED_CSV}")

Removing 10 rows with idx: [22, 181, 146, 257, 274, 113, 247, 198, 91, 141]
original rows : 293
clean rows    : 283  → final_fixed_clean_2.csv
degraded rows : 10  → final_fixed_degraded_2.csv
